# 실험 로그와 자동 실행

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

### 로그 한 행이 완성되는 시점

평가 함수가 질문별 순위와 지표 사전을 반환한 뒤 `json.dump()`가 결과를 파일로 저장합니다. 파일 읽기 성공, 스키마 검사 통과, 지표 개선은 서로 다른 상태입니다. `corpus_fingerprint`와 `metrics_policy` 중 하나라도 다르면 같은 지표 이름의 실행을 직접 비교하기 어렵습니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## Ch 9·10의 실제 검색 로그 찾기

두 챕터가 Q001을 실행해 남긴 새 continuity 로그를 읽습니다. 원래 로그는 수정하지 않습니다.

In [4]:
source_logs = [
    root / "labs/part3/ch09-rag-first-run/runs/continuity-ch09-search.jsonl",
    root / "labs/part3/ch10-document-processing/runs/continuity-ch10-indexing.jsonl",
]
for path in source_logs:
    assert path.exists(), f"앞 챕터 continuity 로그가 없습니다: {path}"
    print(path.relative_to(root), "행:", len(path.read_text(encoding="utf-8").splitlines()))

labs/part3/ch09-rag-first-run/runs/continuity-ch09-search.jsonl 행: 1
labs/part3/ch10-document-processing/runs/continuity-ch10-indexing.jsonl 행: 6


## 검색 결과만 지표로 계산하기

<code>evaluate.py</code>는 질문과 정답 문서 ID가 Ch 11 질의셋과 같은지 검사합니다. Ch 9의 생성 답변은 Ch 15 평가 대상이므로 이 실행에서 점수를 계산하지 않습니다.

In [5]:
import subprocess, sys
from datetime import datetime, timezone

ch11 = root / "labs/part3/ch11-evaluation"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = ch11 / "runs" / f"continuity-upstream-{stamp}.json"
command = [sys.executable, str(ch11 / "evaluate.py"), "--split", "dev"]
for path in source_logs:
    command.extend(["--retrieval-log", str(path)])
command.extend(["--output", str(output)])
completed = subprocess.run(command, cwd=root, check=True, text=True, capture_output=True)
print(completed.stdout.strip())

{
  "run_count": 7,
  "query_counts": [
    1,
    1,
    1,
    1,
    1,
    1,
    1
  ],
  "output": "[사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch11-evaluation/runs/continuity-upstream-20260909T172453430127Z.json"
}


## 실행별 범위와 지표 확인

Q001 하나로 계산한 결과는 연결 형식을 확인합니다. 30개 질문 기준선과 수치를 직접 비교하지 않습니다.

In [6]:
report = json.loads(output.read_text(encoding="utf-8"))
for run in report["runs"]:
    print(run["source_chapter"], run["run_id"], run["query_count"], run["metrics"])
    assert run["query_ids"] == ["Q001"]
print("평가 단위:", report["runs"][0]["evaluation_unit"])
print("새 결과:", output.relative_to(root))

ch09 20260909T161218875711-ch09 1 {'recall_at_logged_candidates': 1.0, 'recall_at_5': None, 'recall_at_50': None, 'precision_at_5': None, 'ndcg_at_5': None, 'logged_candidate_k_min': 2, 'logged_candidate_k_max': 2}
ch10 20260909T172026575849-ch10-01 1 {'recall_at_logged_candidates': 1.0, 'recall_at_5': 1.0, 'recall_at_50': None, 'precision_at_5': 0.2, 'ndcg_at_5': 1.0, 'logged_candidate_k_min': 5, 'logged_candidate_k_max': 5}
ch10 20260909T172026575849-ch10-02 1 {'recall_at_logged_candidates': 1.0, 'recall_at_5': 1.0, 'recall_at_50': None, 'precision_at_5': 0.2, 'ndcg_at_5': 1.0, 'logged_candidate_k_min': 5, 'logged_candidate_k_max': 5}
ch10 20260909T172026575849-ch10-03 1 {'recall_at_logged_candidates': 1.0, 'recall_at_5': 1.0, 'recall_at_50': None, 'precision_at_5': 0.2, 'ndcg_at_5': 1.0, 'logged_candidate_k_min': 5, 'logged_candidate_k_max': 5}
ch10 20260909T172026575849-ch10-04 1 {'recall_at_logged_candidates': 1.0, 'recall_at_5': 1.0, 'recall_at_50': None, 'precision_at_5': 0.2, '